# Uji Kombinasi Atribut terhadap Confidence Level (Kelompok 3)

**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Judul:** Prediksi Risiko Stroke Menggunakan Logistic Regression dan Gradient Boosting dengan Interpretasi Explainable AI

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Bapak meminta satu hal yang belum dijawab notebook lain: **tentukan fitur utama dan
kombinasi atribut terbaik untuk confidence level, bukan akurasi.**

Notebook ini menjawab permintaan itu secara langsung. Beberapa kombinasi atribut diadu
dengan protokol yang sama, lalu diukur **confidence level**-nya.

**Apa yang dimaksud confidence level di sini.** Ukurannya **precision**: dari pasien
yang ditandai berisiko, berapa yang benar-benar stroke. Satu angka ringkasnya adalah
**average precision (AP)**, luas di bawah kurva precision-recall. Alasan memakai
precision dan bukan akurasi ada di Bagian 0.

**Agar tidak tumpang tindih dengan notebook lain.** Notebook `03a` dan `03b` menulis
KNN dan Naive Bayes **dari nol** beserta pembanding CHA2DS2-VASc, notebook `07` mengadu
enam algoritma dan empat strategi, notebook `11` menguji rekayasa fitur dengan uji
berpasangan, dan notebook `12` menghitung selang kepercayaan angka akhir. Notebook ini
tidak mengulang satu pun dari itu. Fokusnya hanya satu: **kombinasi atribut mana yang
memberi confidence level terbaik, dan atribut mana yang paling berpengaruh.**

## 0. Mengapa precision, bukan akurasi

Hanya 4,87% pasien pada dataset ini yang benar-benar stroke (249 dari 5.109). Akibatnya
akurasi menjadi angka yang menyesatkan:

| Model | Akurasi | Recall |
|---|---|---|
| Selalu menjawab "tidak stroke" | 95,1% | 0% |
| Logistic Regression tanpa penyetelan | 95,0% | 0% |

Keduanya tidak menemukan satu pun pasien berisiko, tetapi akurasinya terlihat tinggi.
Karena itu yang diukur di sini adalah **precision**, sejalan dengan catatan paper acuan
bahwa pada kasus seperti ini confidence level lebih tepat diukur dari precision.

**Precision** pada satu titik potong:
$$P = \frac{TP}{TP + FP}$$

**Average precision (AP)**, yaitu luas di bawah kurva precision-recall:
$$\text{AP} = \sum_{n} (R_n - R_{n-1}) \, P_n$$

dengan $P_n$ dan $R_n$ adalah precision dan recall pada titik potong ke-$n$. Mengapa
dipakai AP dan bukan satu titik potong saja: titik potong 0,5 atau 0,8 bisa dipilih
seenaknya, sedangkan AP memakai seluruh titik potong sekaligus, jadi tidak bisa
"diatur" supaya angkanya bagus.

Selain AP, dicatat juga **precision pada recall 0,80**, yaitu titik kerja yang dipakai
notebook `07` (kejar recall 0,80). Angka ini menjawab pertanyaan praktis: kalau kita
menangkap 80% pasien stroke, seberapa yakin kita bahwa yang ditandai memang stroke?

## 1. Persiapan

Data dibaca dari **berkas bersih hasil notebook `02`** (`artifacts/stroke_bersih.csv`)
lewat URL raw GitHub, sama seperti notebook `03a` dan `03b`. Kalau dijalankan lokal
tanpa internet, dibaca dari berkas di sebelah notebook.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
SEED = 42

URL_BERSIH = ("https://raw.githubusercontent.com/ravi-arnan/machine-learning/"
              "main/artifacts/stroke_bersih.csv")
try:
    df = pd.read_csv(URL_BERSIH)
except Exception:
    df = pd.read_csv("../artifacts/stroke_bersih.csv")

y = df["stroke"].astype(int)
X_dasar = pd.get_dummies(df.drop(columns=["stroke"]), drop_first=True).astype(float)

print("Ukuran data bersih   :", df.shape)
print(f"Kasus stroke         : {int(y.sum())} ({y.mean()*100:.2f}%)")
print(f"Fitur dasar (one-hot): {X_dasar.shape[1]}")
df.head(3)

## 2. Kombinasi atribut yang diadu

Atribut dikelompokkan supaya perbandingannya mudah dibaca, lalu disusun menjadi delapan
kombinasi. Kelompok dasarnya sepuluh kolom mentah (lima numerik dan lima kategorikal
yang jadi 15 kolom setelah one-hot).

Diuji juga **fitur turunan if-else** dari laporan Pertemuan 3 (`glukosa_kat`, `bmi_kat`,
`jumlah_faktor`, `risiko_nested`), supaya kombinasi aturan yang sudah dibuat ikut diukur,
bukan hanya diklaim.

In [ ]:
def fitur_turunan(d):
    """Fitur turunan if-else, disalin dari laporan Pertemuan 3. Nilainya ordinal:
    semakin besar semakin berisiko."""
    f = pd.DataFrame(index=d.index)
    f["glukosa_kat"] = np.digitize(d["avg_glucose_level"], [100, 140, 200]).astype(float)
    f["bmi_kat"] = np.digitize(d["bmi"], [18.5, 25, 30]).astype(float)
    f["jumlah_faktor"] = (d["hypertension"] + d["heart_disease"]
                          + (d["avg_glucose_level"] > 200).astype(int)
                          + (d["bmi"] >= 30).astype(int)
                          + (d["age"] >= 60).astype(int)).astype(float)
    f["risiko_nested"] = 0.0
    f.loc[(d.hypertension == 1) | (d.heart_disease == 1), "risiko_nested"] = 1
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1), "risiko_nested"] = 2
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1)
          & (d.avg_glucose_level > 200), "risiko_nested"] = 3
    f.loc[(d.hypertension == 1) & (d.heart_disease == 1)
          & (d.avg_glucose_level > 200) & (d.age > 60), "risiko_nested"] = 4
    return f


TURUNAN = ["glukosa_kat", "bmi_kat", "jumlah_faktor", "risiko_nested"]
X = pd.concat([X_dasar, fitur_turunan(df)], axis=1)

GRUP = {
    "usia (age)": ["age"],
    "hipertensi": ["hypertension"],
    "penyakit jantung": ["heart_disease"],
    "glukosa": ["avg_glucose_level"],
    "bmi": ["bmi"],
    "jenis kelamin": ["gender"],
    "status menikah": ["ever_married"],
    "pekerjaan": ["work_type"],
    "wilayah": ["Residence_type"],
    "merokok": ["smoking_status"],
}
KLINIS = ["usia (age)", "hipertensi", "penyakit jantung", "glukosa"]
SOSIAL = ["jenis kelamin", "status menikah", "pekerjaan", "wilayah", "merokok"]


def kolom(grup):
    """Ubah nama kelompok jadi daftar kolom. Kelompok dasar dicocokkan ke kolom
    one-hot; turunan if-else ditambahkan apa adanya."""
    kol = []
    for g in grup:
        if g == "turunan if-else":
            kol += TURUNAN
        else:
            for k in GRUP[g]:
                kol += [c for c in X_dasar.columns if c == k or c.startswith(k + "_")]
    return list(dict.fromkeys(kol))


KOMBINASI = {
    "A. Usia saja": ["usia (age)"],
    "B. Usia + glukosa": ["usia (age)", "glukosa"],
    "C. Usia + glukosa + hipertensi + bmi": ["usia (age)", "glukosa", "hipertensi", "bmi"],
    "D. Usia + hipertensi + penyakit jantung": ["usia (age)", "hipertensi", "penyakit jantung"],
    "E. Inti klinis": KLINIS,
    "F. Inti klinis + bmi": KLINIS + ["bmi"],
    "G. Semua fitur dasar": KLINIS + ["bmi"] + SOSIAL,
    "H. Semua fitur + turunan if-else": KLINIS + ["bmi"] + SOSIAL + ["turunan if-else"],
}

ringkas = pd.DataFrame(
    [{"Kombinasi": nama, "jumlah fitur": len(kolom(grup))} for nama, grup in KOMBINASI.items()]
)
ringkas

## 3. Cara mengukur

**Model.** Ukuran utama memakai **Logistic Regression** karena ia model di judul
proyek, deterministik, dan probabilitasnya terkalibrasi secara alami (temuan notebook
`11`). Di Bagian 7 pengukuran diulang dengan tiga model lain untuk memeriksa apakah
kesimpulannya berubah.

**Protokol adil.** Semua kombinasi diuji pada baris dan lipatan cross-validation yang
persis sama: `StratifiedKFold(5)`. Kalau barisnya berbeda, selisihnya bukan pengaruh
atribut melainkan pengaruh datanya. Probabilitas diambil **out-of-fold**, artinya tiap
pasien dinilai oleh model yang tidak pernah melihatnya saat latih.

**Ketidakpastian.** Karena hanya ada 249 kasus stroke, satu angka AP saja bisa
menyesatkan. Setiap AP diberi **selang kepercayaan bootstrap 95%**: baris data
diambil ulang acak sebanyak 2.000 kali, AP dihitung tiap kali, lalu diambil persentil
ke-2,5 dan ke-97,5. Cara yang sama dipakai notebook `12` untuk angka akhir.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)


def model_acuan():
    """Logistic Regression dengan hyperparameter hasil GridSearch notebook 07."""
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.1, penalty="l1", solver="liblinear",
                           max_iter=5000, random_state=SEED),
    )


def skor_oof(model, Xk, y, cv=cv):
    """Probabilitas out-of-fold kelas stroke."""
    return cross_val_predict(model, Xk, y, cv=cv, method="predict_proba")[:, 1]


def presisi_pada_recall(y_true, p, target=0.80):
    """Precision terbesar yang masih mencapai recall >= target."""
    pres, rec, _ = precision_recall_curve(y_true, p)
    idx = np.where(rec >= target)[0]
    return float(pres[idx].max()) if len(idx) else float("nan")


def bootstrap_ap(y_true, p, B=2000, seed=SEED):
    """Selang kepercayaan 95% untuk average precision."""
    rng = np.random.default_rng(seed)
    yt = np.asarray(y_true)
    n = len(yt)
    nilai = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if yt[idx].sum() == 0:
            continue
        nilai.append(average_precision_score(yt[idx], p[idx]))
    return float(np.percentile(nilai, 2.5)), float(np.percentile(nilai, 97.5))

## 4. Confidence level tiap kombinasi atribut

Semua kombinasi diukur dengan model dan protokol yang sama. Diurutkan dari AP terbesar.

In [ ]:
baris = []
for nama, grup in KOMBINASI.items():
    kol = kolom(grup)
    p = skor_oof(model_acuan(), X[kol], y)
    ap = average_precision_score(y, p)
    lo, hi = bootstrap_ap(y, p)
    baris.append({
        "Kombinasi atribut": nama,
        "jumlah fitur": len(kol),
        "AP": round(ap, 4),
        "AP 2,5%": round(lo, 4),
        "AP 97,5%": round(hi, 4),
        "precision @ recall 0,80": round(presisi_pada_recall(y, p), 4),
        "ROC-AUC": round(roc_auc_score(y, p), 4),
    })

tabel = pd.DataFrame(baris).sort_values("AP", ascending=False).reset_index(drop=True)
tabel

**Yang terbaca dari tabel.**

1. **Kombinasi terbaik adalah inti klinis** (`age`, `hipertensi`, `penyakit jantung`,
   `glukosa`) dengan AP **0,184**. Menambah `bmi` (kombinasi F) **tidak mengubah apa
   pun**: AP-nya sama persis 0,184, tanda kontribusi `bmi` nol setelah fitur lain ada.
2. **Menambah semua fitur justru tidak menolong.** Kombinasi G (15 fitur) AP 0,183 dan
   H (19 fitur, dengan turunan if-else) malah turun ke 0,179.
3. **Selang kepercayaan semua kombinasi saling bertindihan lebar** (misalnya inti
   klinis 0,155 sampai 0,224). Artinya perbedaan 0,001 sampai 0,004 di antara kombinasi
   besar **tidak boleh diklaim bermakna**. Yang boleh diklaim hanya bedanya dengan
   kombinasi yang jauh lebih ramping di bawahnya.
4. **Dua atribut sudah membawa sebagian besar sinyal.** "Usia saja" AP 0,164 dan
   "usia + glukosa" AP 0,181, sudah mendekati angka kombinasi lengkap.

In [ ]:
pilih = ["A. Usia saja", "E. Inti klinis", "G. Semua fitur dasar",
         "H. Semua fitur + turunan if-else"]

fig, ax = plt.subplots(figsize=(7, 5.5))
for nama in pilih:
    p = skor_oof(model_acuan(), X[kolom(KOMBINASI[nama])], y)
    pres, rec, _ = precision_recall_curve(y, p)
    ax.plot(rec, pres, lw=2,
            label=f"{nama} (AP {average_precision_score(y, p):.3f})")
ax.axhline(y.mean(), color="grey", ls="--", lw=1,
           label=f"tebakan acak (prevalensi {y.mean():.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Kurva precision-recall tiap kombinasi atribut")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Kurva-kurvanya saling menempel. Kombinasi ramping (usia saja) terlihat sedikit di bawah,
tetapi sisanya nyaris berimpit, persis seperti yang dikatakan selang kepercayaan di
tabel sebelumnya.

## 5. Ablation: membuang satu kelompok atribut

Tabel di atas membandingkan kombinasi yang disusun manusia. Cara yang lebih langsung
menjawab "atribut mana yang paling berpengaruh" adalah **ablation**: mulai dari semua
fitur dasar, lalu buang satu kelompok atribut, dan lihat berapa AP yang hilang.
Kelompok yang paling merugikan saat dibuang itulah yang paling berpengaruh.

In [ ]:
GROUPS = list(GRUP.keys())
ap_penuh = average_precision_score(y, skor_oof(model_acuan(), X[kolom(GROUPS)], y))

abl = [{"dibuang": "(tidak ada)", "AP": round(ap_penuh, 4), "selisih AP": 0.0}]
for g in GROUPS:
    p = skor_oof(model_acuan(), X[kolom([x for x in GROUPS if x != g])], y)
    ap = average_precision_score(y, p)
    abl.append({"dibuang": g, "AP": round(ap, 4), "selisih AP": round(ap - ap_penuh, 4)})

abl = pd.DataFrame(abl)
print(f"AP semua fitur dasar: {ap_penuh:.4f}\n")
abl

**Usia adalah atribut yang paling berpengaruh, jauh di atas yang lain.** Membuangnya
menjatuhkan AP dari 0,183 ke 0,125, selisih **-0,058**, sekitar sepuluh kali lipat
selisih atribut berikutnya. Glukosa menyusul di tempat kedua (selisih -0,006), lalu
pekerjaan (-0,004).

Sebaliknya, empat kelompok malah **menaikkan** AP saat dibuang (selisihnya positif):
`status menikah` (+0,005), `penyakit jantung` (+0,002), `jenis kelamin` (+0,001), dan
`merokok` (+0,0006). Artinya dengan fitur lain sudah ada, keempatnya cenderung menjadi
derau, bukan sinyal. Besarnya pun di dalam derau, jadi jangan dibaca sebagai
"membuangnya pasti lebih baik", hanya sebagai "tidak ada bukti ia menambah apa pun".

Bagian berikutnya memakai gambar supaya lebih cepat terbaca.

In [ ]:
urut = abl[abl["dibuang"] != "(tidak ada)"].set_index("dibuang")["selisih AP"].sort_values()
warna = ["#4C78A8" if v < 0 else "#E45756" for v in urut]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(urut.index, urut.values, color=warna)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Perubahan AP saat kelompok atribut dibuang")
ax.set_title("Usia jauh paling menentukan, sisanya hampir nol")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Seleksi maju: urutan pengaruh atribut

Ablation membuang dari himpunan penuh. Kebalikannya, **seleksi maju** mulai dari kosong
lalu menambah satu kelompok yang paling menaikkan AP, berulang sampai habis. Urutan
penambahannya memperlihatkan atribut mana yang paling awal dibutuhkan model.

> **Peringatan metodologis.** Seleksi maju memilih atribut memakai data yang sama yang
> dipakai menilainya, jadi angkanya **optimistis** dan cenderung memilih fitur yang
> kebetulan cocok dengan derau. Hasilnya harus dibaca sebagai **petunjuk**, bukan
> sebagai kombinasi yang divalidasi. Kombinasi final tetap diambil dari Bagian 4 yang
> protokolnya sudah bebas pilih-pilih.

In [ ]:
terpilih, sisa, riwayat = [], list(GRUP.keys()), []
while sisa:
    kandidat = []
    for g in sisa:
        p = skor_oof(model_acuan(), X[kolom(terpilih + [g])], y)
        kandidat.append((average_precision_score(y, p), g))
    ap, g = max(kandidat)
    terpilih.append(g)
    sisa.remove(g)
    riwayat.append({"ditambahkan": g, "AP": round(ap, 4)})

pd.DataFrame(riwayat)

In [ ]:
urutan = pd.DataFrame(riwayat)
ap_klinis = average_precision_score(
    y, skor_oof(model_acuan(), X[kolom(KOMBINASI["E. Inti klinis"])], y))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(range(1, len(urutan) + 1), urutan["AP"], "o-", color="#4C78A8", lw=2)
for i, r in urutan.iterrows():
    ax.annotate(r["ditambahkan"], (i + 1, r["AP"]), fontsize=7,
                xytext=(0, 6), textcoords="offset points", ha="center")
ax.axhline(ap_klinis, color="#E45756", ls="--", lw=1,
           label=f"AP inti klinis ({ap_klinis:.3f})")
ax.set_xlabel("Jumlah kelompok atribut ditambahkan")
ax.set_ylabel("Average precision")
ax.set_title("Seleksi maju: AP mentok setelah usia dan glukosa")
ax.legend()
plt.tight_layout()
plt.show()

**Kurvanya mentok cepat.** Dua langkah pertama (usia, lalu glukosa) sudah membawa AP ke
0,181. Penambahan ketiga sampai kesembilan hanya bergerak di kisaran 0,188 dan
akhimya turun lagi ke 0,183 saat semua masuk. Yang menaikkan setelah glukosa
(`pekerjaan`, `bmi`, `wilayah`) besarnya di dalam derau, jadi tidak layak dipercaya.

Kesimpulan gabungan dari ablation dan seleksi maju sama: **usia paling berpengaruh,
glukosa kedua, sisanya hampir tidak berarti.**

## 7. Ketahanan di model lain: apakah kesimpulannya berubah?

Seluruh pengukuran di atas memakai Logistic Regression. Pertanyaan wajar: apakah
"kombinasi terbaik" itu kesimpulan tentang atributnya atau cuma tentang modelnya? Karena
itu tiga kombinasi kunci diukur ulang dengan Gradient Boosting, KNN, dan Gaussian Naive
Bayes, model yang sama seperti yang diuji manual di notebook `03a` dan `03b`.

In [ ]:
MODEL = {
    "Logistic Regression": model_acuan(),
    "Gradient Boosting": make_pipeline(
        StandardScaler(), GradientBoostingClassifier(random_state=SEED)),
    "KNN (k=15)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "Gaussian Naive Bayes": make_pipeline(StandardScaler(), GaussianNB()),
}

uji = ["E. Inti klinis", "G. Semua fitur dasar", "H. Semua fitur + turunan if-else"]
baris = []
for nama_model, m in MODEL.items():
    for nama_kombinasi in uji:
        p = skor_oof(m, X[kolom(KOMBINASI[nama_kombinasi])], y)
        baris.append({
            "Model": nama_model,
            "Kombinasi atribut": nama_kombinasi,
            "AP": round(average_precision_score(y, p), 4),
            "precision @ recall 0,80": round(presisi_pada_recall(y, p), 4),
            "ROC-AUC": round(roc_auc_score(y, p), 4),
        })

lintas = pd.DataFrame(baris)
lintas.pivot(index="Kombinasi atribut", columns="Model", values="AP")

**Urutannya bergantung model, tetapi pesan utamanya tidak berubah.** Model dengan
inductive bias berbeda menyukai kombinasi berbeda:

| Model | Kombinasi terbaik | AP |
|---|---|---|
| Logistic Regression | Inti klinis | 0,184 |
| Gaussian Naive Bayes | Inti klinis | 0,166 |
| Gradient Boosting | Semua fitur + turunan if-else | 0,158 |
| KNN (k=15) | Inti klinis | 0,118 |

Dua hal yang konsisten di semua model. Pertama, **Logistic Regression tetap yang
terbaik** pada kombinasi ramping, sesuai temuan notebook `11` bahwa model sederhana
menang pada data ini. Kedua, **KNN paling buruk** di semua kombinasi, sejalan dengan
hasil manual di notebook `03a` yang menunjukkan KNN gagal karena skala fitur dan kelas
yang sangat tidak seimbang.

Menariknya Gradient Boosting justru paling tinggi dengan seluruh fitur, termasuk
turunan if-else. Namun bedanya dengan inti klinis (0,158 lawan 0,152) masih di dalam
derau, jadi ini lebih tepat dibaca sebagai "Gradient Boosting sanggup memanfaatkan
fitur tambahan" daripada "fitur tambahan itu penting".

## 8. Kesimpulan

**Pertanyaan yang dijawab:** kombinasi atribut mana yang memberi confidence level
terbaik, dan atribut mana yang paling berpengaruh?

**Jawabannya.**

1. **Kombinasi dengan confidence terbaik adalah inti klinis:** `age`, `hipertensi`,
   `penyakit jantung`, `avg_glucose_level`, dengan AP 0,184 (selang 95%: 0,155 sampai
   0,224). Menambah `bmi` tidak mengubah apa pun.
2. **Atribut paling berpengaruh adalah usia (`age`), jauh di atas yang lain.** Dibuang,
   AP anjlok 0,183 ke 0,125. Glukosa menyusul kedua, sisanya hampir nol.
3. **Menambah lebih banyak atribut tidak menambah confidence.** Semua fitur dasar (AP
   0,183) dan semua fitur plus turunan if-else (AP 0,179) tidak lebih baik daripada
   kombinasi ramping. Ini menjelaskan mengapa rekayasa fitur di notebook `11`
   eksperimennya gagal.
4. **Kesimpulan ini bukan cuma tentang modelnya.** Logistik Regression dan Naive Bayes
   sama-sama memilih inti klinis. Gradient Boosting sedikit menyukai fitur lebih banyak,
   tetapi bedanya di dalam derau.

**Hubungan dengan notebook lain.** Hasil ini sejalan dengan notebook `01`, yang
menempatkan `age` paling kuat secara korelasi dan mutual information, dan dengan
notebook `09`, yang lewat SHAP menempatkan `age` dan `avg_glucose_level` di puncak
kontribusi. Tiga cara berbeda, satu kesimpulan, dan itu tanda kesimpulannya kokoh.

**Batasan yang harus dibaca bersama angka di atas.**

- Hanya ada 249 kasus stroke, sehingga selang kepercayaan setiap kombinasi lebar dan
  saling bertindihan. Perbedaan antar kombinasi besar **tidak boleh** diklaim bermakna;
  yang boleh diklaim hanya bedanya dengan kombinasi yang jauh lebih ramping.
- Seleksi maju di Bagian 6 memilih fitur memakai data yang sama, jadi angkanya
  optimistis dan tidak dipakai sebagai keputusan final.
- "Confidence level" di sini diukur dari precision. Kalau diukur dari kalibrasi
  probabilitas, hasilnya bisa berbeda, dan itu bisa menjadi kelanjutan sendiri.